# 全国お天気マップアプリ

- **氏名**: 田上達貴 兼平祥汰　白石大策
- **問い**: 日本全国の天気と気温差を一目で直感的に比較するにはどうすればいいか？
- **工夫した点**: `for` 文で主要11都市の天気APIデータを自動取得し、テレビの天気予報のようなカード形式で日本地図上に可視化した。

In [1]:
import json
from urllib.request import urlopen
import folium

# ① 都市リスト（名前, 緯度, 経度）
cities = [
    ["札幌", 43.06, 141.35],
    ["仙台", 38.27, 140.87],
    ["東京", 35.69, 139.69],
    ["柏",   35.86, 139.98],
    ["新潟", 37.92, 139.04],
    ["金沢", 36.56, 136.66],
    ["名古屋", 35.18, 136.91],
    ["大阪", 34.69, 135.50],
    ["広島", 34.39, 132.46],
    ["福岡", 33.59, 130.40],
    ["那覇", 26.21, 127.68],
]

# ② 日本地図を作る（国土地理院の地図タイルを使用・APIキー不要）
m = folium.Map(
    location=[35.5, 136.0],
    zoom_start=5,
    tiles="https://cyberjapandata.gsi.go.jp/xyz/pale/{z}/{x}/{y}.png",
    attr="国土地理院"
)

# ③ for文で1都市ずつ天気を調べて、地図にカードを貼る
for name, lat, lon in cities:
    # APIから天気データを取得
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&daily=weather_code,temperature_2m_max,temperature_2m_min&forecast_days=1&timezone=auto"
    data = json.load(urlopen(url))["daily"]

    code = data["weather_code"][0]
    tmax = round(data["temperature_2m_max"][0])
    tmin = round(data["temperature_2m_min"][0])

    # 天気コードを絵文字に変換（0は晴れ、1〜3はくもり、それ以外は雨・雪）
    if code == 0:
        icon = "☀️"
    elif code <= 3:
        icon = "⛅"
    else:
        icon = "☔"

    # 天気カードのデザイン（都市名・絵文字・最高/最低気温）
    card = f"""
    <div style="background:white; border:2px solid #333; border-radius:6px; width:64px; text-align:center; font-weight:bold; font-size:11px;">
        <div style="background:#eee;">{name}</div>
        <div style="font-size:18px;">{icon}</div>
        <div><span style="color:red;">{tmax}℃</span> / <span style="color:blue;">{tmin}℃</span></div>
    </div>
    """

    # 地図の指定場所にカードを貼る
    folium.Marker([lat, lon], icon=folium.DivIcon(html=card)).add_to(m)

m